# EDA & Preparing the dataset for hippocampus segmentation

In this notebook I will prepare the hippocampus dataset using Python, and will do an exploratory data analysis of the dataset.

## Import libraries

In [ ]:
# Import the following libraries that we will use: nibabel, matplotlib, numpy
import numpy as np
from PIL import Image
import os
import shutil
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as nd
from pathlib import Path

## Loading NIFTI images using NiBabel

NiBabel is a python library for working with neuro-imaging formats (including NIFTI). Our volumes and labels are in NIFTI format, so we will use nibabel to load and inspect them.

NiBabel documentation could be found here: https://nipy.org/nibabel/

Our dataset sits in two directories - *images* and *labels*. Each image is represented by a single file (we are fortunate to have our data converted to NIFTI) and has a corresponding label file which is named the same as the image file.

In [ ]:
# Data sits in data/TrainingSet by default (can override via HIPPO_DATA_DIR environment variable).
# Load an image and a segmentation mask into variables called image and label
default_data_path = Path.cwd() / "data" / "TrainingSet"
data_path = Path(os.getenv("HIPPO_DATA_DIR", default_data_path)).resolve()
image = nib.load(str(data_path / "images" / "hippocampus_042.nii.gz"))
label = nib.load(str(data_path / "labels" / "hippocampus_042.nii.gz"))
print(f"Using dataset path: {data_path}")

In [ ]:
# Check for outliers/bad data by checking volume dimensions of all images
images_path = str(data_path / "images")
labels_path = str(data_path / "labels")

def show_volume_dimensions(directory):
    print("Directory: " + directory)
    # Get directory object
    directory_object = os.fsencode(directory)
    # For each file in directory
    for file in os.listdir(directory_object):
        # Get filename
        filename = os.fsdecode(file)
        # Load nifti file volume
        volume = nib.load(os.path.join(directory, filename))
        # Print filename and volume dimensions
        print(filename)
        print(volume.shape)

# Print dimensions of all image volumes
show_volume_dimensions(images_path)

Ok so there is a bit of variance in the image volume dimensions. However for volume hippocampus_010.nii.gz there is a huge difference in dimensions (512, 512, 241) so this looks like an outlier.

In [ ]:
# Print dimensions of all label volumes
show_volume_dimensions(labels_path)

Ok so there is a bit of variance in the label volume dimensions. However for volume hippocampus_281.nii.gz there is a huge difference in dimensions (512, 512, 94) so this looks like an outlier.

In [ ]:
# Nibabel can present image data as a Numpy array by calling the method get_fdata()
# The array will contain a multi-dimensional Numpy array with numerical values representing voxel intensities. 
# In our case, images and labels are 3-dimensional, so get_fdata will return a 3-dimensional array. We can verify this
# by accessing the .shape attribute. Check the dimensions of the input arrays

# Visualize a few slices from the dataset, along with their labels. 
# We can adjust plot sizes like so if they are too small:

# Get images data
image_data = image.get_fdata()
label_data = label.get_fdata()

# Print dimensions
print(image_data.shape)
print(label_data.shape)

In [ ]:
def show_slices(slices):
    """ Function to display row of image slices """
    fig, axes = plt.subplots(1, len(slices))
    for i, slice in enumerate(slices):
        axes[i].imshow(slice.T, cmap="gray", origin="lower")

# Take center slice in each plane
# Saggital plane (split eyes) center slice
slice_0 = image_data[18, :, :]
# Coronal plane (face on) center slice
slice_1 = image_data[:, 26, :]
# Axial/Transverse plane (top down) center slice
slice_2 = image_data[:, :, 17]
    
show_slices([slice_0, slice_1, slice_2])
plt.suptitle("Center slices for saggital, coronal and axial planes")  

In [ ]:
plt.rcParams["figure.figsize"] = (15,15)

def show_image_and_label_slices(slice_index):
    """ Function to display for given value slice index, image and label """
    fig, axes = plt.subplots(1, 2)
    # Plot axial plane slice & label
    slice_image = image_data[:, :, slice_index]
    slice_labels = label_data[:, :, slice_index]
    axes[0].imshow(slice_image.T, cmap="gray", origin="lower")
    axes[0].set_title('Image')
    axes[1].imshow(slice_labels.T, cmap="gray", origin="lower")
    axes[1].set_title('Labels')

# Show slice 10 image and labels
show_image_and_label_slices(10)

In [ ]:
# Show slice 30 image and labels
show_image_and_label_slices(20)

In [ ]:
# Show slice 40 image and labels
show_image_and_label_slices(30)

I notice from my label slices there seem 2 shades of labels one grey and one white? do we have 2 labels? i.e. 3 states, no label, label 1 & label 2? lets test that and see what labels we have.

In [ ]:
np.unique(label_data)

In [ ]:
# Orthographic projection on axial plane
vr = np.zeros((image_data.shape[0], image_data.shape[1]))

for z in range (image_data.shape[2]):
    vr += image_data[:,:,z]
    
plt.imshow(nd.rotate(vr, 90), cmap="gray")

In [ ]:
# Maximum intensity projection on axial plane
# For a change, let's stack slices along the Y axis and thus visualize the coronal plane
mip = np.zeros((image_data.shape[0], image_data.shape[1]))

for z in range(image_data.shape[2]):
    mip = np.maximum(mip, image_data[:,:,z])
    
plt.imshow(nd.rotate(mip, 90), cmap="gray")

## Looking at single image data
In this section we will look closer at the NIFTI representation of our volumes. In order to measure the physical volume of hippocampi, we need to understand the relationship between the sizes of our voxels and the physical world.

In [ ]:
# Nibabel supports many imaging formats, NIFTI being just one of them. 
# Check the format of our images?
image.header_class

In [ ]:
print(image.header)

Further down we will be inspecting .header attribute that provides access to NIFTI metadata. We can use this resource as a reference for various fields: https://brainder.org/2012/09/23/the-nifti-file-format/

In [ ]:
# How many bits per pixel are used?
image.header['bitpix']

In [ ]:
# What are the units of measurement?
# Units of pixdim - the dimension of the grid spacing in voxels - are encoded as binary not decimal numbers!
image.header['xyzt_units']

In [ ]:
# Use convenience function to extract units from binary encoded field
image.header.get_xyzt_units()

In [ ]:
# Do we have a regular grid? What are grid spacings?
image.header['pixdim']

So pixdim dimensions 1-3 correspond to the grid spacings for the x,y & z dimensions. As these are all equal to 1 - we do have a regular grid.

In [ ]:
# What dimensions represent axial, sagittal, and coronal slices? How do we know?

# In the documentation here: https://brainder.org/2012/09/23/the-nifti-file-format/
# It says "Predefined dimensions for space and time: In the nifti format, the first three dimensions are reserved 
# to define the three spatial dimensions — x, y and z —"

# Take center slice in each plane
# Saggital plane (split eyes) center slice
slice_0 = image_data[18, :, :]
# Coronal plane (face on) center slice
slice_1 = image_data[:, 26, :]
# Axial/Transverse plane (top down) center slice
slice_2 = image_data[:, :, 17]
    
show_slices([slice_0, slice_1, slice_2])
plt.suptitle("Center slices for saggital, coronal and axial planes")  

In [ ]:
# We should have enough information now to decide what are dimensions of a single voxel
# Compute the volume (in mm³) of a hippocampus using one of the labels you've loaded. 
# We should get a number between ~2200 and ~4500

volume = np.sum(label.get_fdata() > 0)
volume

## Plotting some charts

In [ ]:
# Plot a histogram of all volumes that we have in our dataset and see how 
# our dataset measures against a slice of a normal population represented by the chart below.

def plot_label_volumes():
    
    volume_totals = []
    # Get directory object for labels
    directory_object = os.fsencode(labels_path)
    # For each label in directory
    for file in os.listdir(directory_object):
        # Get filename
        filename = os.fsdecode(file)
        # Load nifti file for label
        label = nib.load(labels_path + filename)
        # Calculate label volume
        label_volume = np.sum(label.get_fdata() > 0)
        # Append volume
        volume_totals.append(label_volume)
        
    plt.hist(volume_totals)
        
plot_label_volumes()

We can observe an outlier with a much bigger volume of 20,000. We identified potential outliers earlier in our process. We can remove these so we can focus on the bulk of the distribution.

In [ ]:
def plot_label_volumes2():
    
    volume_totals = []
    # Get directory object for labels
    directory_object = os.fsencode(labels_path)
    # For each label in directory
    for file in os.listdir(directory_object):
        # Get filename
        filename = os.fsdecode(file)
        # Load nifti file for label
        label = nib.load(labels_path + filename)
        # Calculate label volume
        label_volume = np.sum(label.get_fdata() > 0)
        # If we detect an outlier
        if (label_volume > 7500):
            # Print message
            print("Outlier detected with volume of " + str(label_volume) + " in file " + filename)
            print("Outlier excluded from plot")
        else:
            # Append volume
            volume_totals.append(label_volume)
        
    plt.hist(volume_totals)
        
plot_label_volumes2()

<img src="img/nomogram_fem_right.svg" width=400 align=left>

**I observed an outlier at around 20,000 which we removed to re-plot the histogram more clearly. With outliers removed, my distribution of values for the right-hippocampus is similar to that in the chart of the normal population.**

In the real world we would have precise information about the ages and conditions of our patients, and understanding how our dataset measures against population norm would be the integral part of clinical validation that we talked about in last lesson. Unfortunately, we do not have this information about this dataset, so we can only guess why it measures the way it is.

The mask seems to have two classes, labeled with values `1` and `2` respectively. When we visualized sagittal or axial views, we got a guess of what those are. Class 1 is the anterior segment of the hippocampus and class 2 is the posterior one. 

For the purpose of volume calculation we do not care about the distinction, however we will still train our network to differentiate between these two classes and the background

In [ ]:
# Copy the clean dataset to out/TrainingSet. We will use it for training.
# Note 118 included as has image but no corresponding label
outliers = ['hippocampus_010.nii.gz', 'hippocampus_281.nii.gz', 'hippocampus_118.nii.gz']

def copy_clean_dataset():
    base_out = Path("out") / "TrainingSet"
    clean_labels_path = base_out / "labels"
    clean_images_path = base_out / "images"
    clean_labels_path.mkdir(parents=True, exist_ok=True)
    clean_images_path.mkdir(parents=True, exist_ok=True)

    # LABELS
    directory_object = os.fsencode(labels_path)
    for file in os.listdir(directory_object):
        filename = os.fsdecode(file)
        if filename not in outliers:
            shutil.copy2(Path(labels_path) / filename, clean_labels_path / filename)

    # IMAGES
    directory_object = os.fsencode(images_path)
    for file in os.listdir(directory_object):
        filename = os.fsdecode(file)
        if filename not in outliers:
            shutil.copy2(Path(images_path) / filename, clean_images_path / filename)

    print(f"Clean dataset copied to: {base_out.resolve()}")

copy_clean_dataset()

## Conclusion

In this section we have inspected a dataset of MRI scans and related segmentations, represented as NIFTI files. We have visualized some slices, and understood the layout of the data. We have also inspected file headers to understand what how the image dimensions relate to the physical world and we have understood how to measure our volume. We have then inspected dataset for outliers, and have created a clean set that is ready for consumption by our ML algorithm. 

In the next section we will create training and testing pipelines for a UNet-based machine learning model, run and monitor the execution, and will produce test metrics. This will give us all we need to use the model in the clinical context and reason about its performance.